# 仮説検証プロジェクト — 前日の A〜E 地点の天気から F 地点の天気を予測できるか

## 1. 研究課題

A、B、C、D、E、F という 6 地点について、各日の天気を二値変数で表します(`1` = 快晴、`0` = 快晴ではない)。

検証したい仮説:

> **ある日の F 地点の天気は、前日の A〜E 地点の天気のパターンによって決まる。**

そこで予測モデルの入力と目的変数を次のように設定します。

- 入力: 前日 $t-1$ の A〜E 地点の天気 $X_t = (A_{t-1}, B_{t-1}, C_{t-1}, D_{t-1}, E_{t-1})$
- 目的変数: 当日 $t$ の F 地点の天気 $y_t = F_t$

このノートブックは**自己完結**です。外部データや画像は使わず、再現可能なダミーデータを内部で生成し、
CPU だけで実行できる規模のモデル(scikit-learn)で検証します。

> ⚠️ **最初に明記しておくべきこと**: ダミーデータによる分析は「**分析方法が正しく動くことの確認**」です。
> ここで良い結果が出ても、**現実の天気に関する仮説が実証されたことにはなりません**(最終章で再度述べます)。

**最初のセルは日本語フォントの読み込みのため、実行に数十秒かかります。**上から順に実行してください。

In [ ]:
import piplite
await piplite.install("matplotlib-fontja==1.1.0")

import os
import warnings

import matplotlib_fontja
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

matplotlib_fontja.japanize()
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)

SEED = 42
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

## 2. ダミーデータの生成

約 365 日分の時系列データを生成します(乱数シード `42` に固定)。
現実の天気らしくするため、A〜E 地点の天気には次の 3 要素を入れます。

1. **全地点に影響する共通の天候要因** … その日の「地域全体の天候の良さ」$c_t$(前日を引きずる AR(1) 過程)
2. **各地点固有のランダム要因** … 地点ごとの独立なノイズ
3. **時間的自己相関** … 各地点とも、前日快晴なら翌日も快晴になりやすい持続項

F 地点の当日の天気は、**前日の A〜E の値**から次の潜在スコアで生成します。
線形項に加えて **A×C、B×E の交互作用**を含む非線形な確率モデルです(ニューラルネットワークを使う意味を持たせるため)。

```
score = -4.0 + 1.2 * (A+B+C+D+E) + 1.5 * A*C + 1.0 * B*E - 0.8 * D   (すべて前日の値)
確率 = sigmoid(score) に従って F を 0/1 で生成
```

この「正解の生成規則」は後の章(32 パターン分析・仮説検証)で、モデルが復元できたかの答え合わせに使います。

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def f_score_formula(a, b, c, d, e):
    """F 地点の潜在スコア(引数はすべて前日の A〜E)。"""
    return (
        -4.0
        + 1.2 * (a + b + c + d + e)
        + 1.5 * a * c
        + 1.0 * b * e
        - 0.8 * d
    )


def generate_weather(n_days=365, seed=SEED):
    rng = np.random.default_rng(seed)
    sites = ["A", "B", "C", "D", "E"]
    X = np.zeros((n_days, 5), dtype=int)
    F = np.zeros(n_days, dtype=int)

    common = 0.0                                # 共通の天候要因(AR(1))
    prev = rng.binomial(1, 0.5, size=5)         # 初日の前日相当

    for t in range(n_days):
        common = 0.6 * common + rng.normal(0, 0.8)
        for i in range(5):
            z = (
                0.9 * common                     # 共通要因
                + 0.8 * (2 * prev[i] - 1)        # 前日からの持続
                + rng.normal(0, 0.7)             # 地点固有のランダム要因
            )
            X[t, i] = rng.binomial(1, sigmoid(z))
        if t == 0:
            F[t] = rng.binomial(1, 0.4)          # 初日のみ前日が無いため仮の値(後で行ごと削除)
        else:
            p = sigmoid(f_score_formula(*X[t - 1]))
            F[t] = rng.binomial(1, p)
        prev = X[t]

    df = pd.DataFrame(X, columns=sites)
    df.insert(0, "day", np.arange(1, n_days + 1))
    df["F"] = F
    return df


weather = generate_weather()
weather.to_csv("data/dummy_weather.csv", index=False)
print("保存しました: data/dummy_weather.csv", weather.shape)
weather.head()

生成したデータの性質を確認します。**F の快晴率が 25〜60% に収まっているか**、
そして「前日快晴なら翌日も快晴になりやすい」自己相関が入っているかを見ます。

In [ ]:
print("各地点の快晴率:")
print(weather[["A", "B", "C", "D", "E", "F"]].mean().round(3))

print()
for site in ["A", "B", "C", "D", "E"]:
    today = weather[site].iloc[1:].to_numpy()
    yesterday = weather[site].iloc[:-1].to_numpy()
    p1 = today[yesterday == 1].mean()
    p0 = today[yesterday == 0].mean()
    print(f"{site}: P(快晴 | 前日快晴) = {p1:.2f}   P(快晴 | 前日曇雨) = {p0:.2f}")

In [ ]:
shown = 120
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.imshow(
    weather[["A", "B", "C", "D", "E", "F"]].iloc[:shown].T,
    cmap="Blues", aspect="auto", interpolation="nearest",
)
ax.set_yticks(range(6), ["A", "B", "C", "D", "E", "F"])
ax.set_xlabel("日")
ax.set_title(f"最初の {shown} 日の天気(濃い = 快晴)— 縦の筋は共通要因、横の連なりは持続性")
plt.tight_layout()
plt.show()

## 3. 時系列の前処理とデータ分割

`shift(1)` で **前日の A〜E** を説明変数 `A_lag1`〜`E_lag1` として明示的に作り、先頭の欠損行を削除します。

**データ漏洩(リーク)を防ぐためのルール**を先に確認しておきます。

| してはいけないこと | このノートブックでの対策 |
|---|---|
| 当日の A〜E で当日の F を予測する | 説明変数は `shift(1)` した前日値のみ |
| 将来日の情報を学習に含める | 日付順を保ったまま前から 70/15/15% に分割(シャッフルしない) |
| 全データで前処理を学習してから分割する | 特徴量は 0/1 のみで、標準化などの「学習する前処理」を使わない |
| テストデータでモデルや閾値を調整する | ハイパーパラメータと分類閾値は**検証データのみ**で選ぶ |

In [ ]:
features = ["A_lag1", "B_lag1", "C_lag1", "D_lag1", "E_lag1"]

df = weather.copy()
for site in ["A", "B", "C", "D", "E"]:
    df[f"{site}_lag1"] = df[site].shift(1)
df = df.dropna().reset_index(drop=True)

n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

train = df.iloc[:n_train]
val = df.iloc[n_train:n_train + n_val]
test = df.iloc[n_train + n_val:]

X_train, y_train = train[features].to_numpy(), train["F"].to_numpy()
X_val, y_val = val[features].to_numpy(), val["F"].to_numpy()
X_test, y_test = test[features].to_numpy(), test["F"].to_numpy()

for name, part, y in [("学習", train, y_train), ("検証", val, y_val), ("テスト", test, y_test)]:
    print(f"{name}: {len(part):3d} 日 (day {part['day'].iloc[0]:3.0f}〜{part['day'].iloc[-1]:3.0f}) "
          f" F の快晴率 {y.mean():.2f}")

## 4. モデルの学習

次の 5 つのモデルを、**同一の学習データ**で学習させ、後で**同一のテストデータ**で評価します。

1. **多数派モデル** … 常に学習データの多数派クラスを予測(確率は学習データの快晴率で固定)。これに勝てなければ話になりません
2. **ロジスティック回帰** … 線形モデルの代表
3. **決定木** … 単純な非線形モデル(深さは検証データで選択)
4. **ランダムフォレスト** … 決定木のアンサンブル
5. **ニューラルネットワーク** … 多層パーセプトロン(隠れ層の構成を検証データで選択)

「非線形モデル(3〜5)が線形モデル(2)より良くなるか」が仮説検証の重要な比較ポイントです。

In [ ]:
models = {}          # name -> テストデータでの予測確率(あとで一括評価)
proba_val_nn = None  # NN の検証データ予測(閾値調整で使用)

# --- 1. 多数派モデル ---
majority_class = int(y_train.mean() >= 0.5)
base_rate = y_train.mean()
models["多数派モデル"] = {
    "proba": np.full(len(y_test), base_rate),
    "pred": np.full(len(y_test), majority_class),
}
print(f"学習データの多数派クラス: {majority_class} (快晴率 {base_rate:.2f})")

# --- 2. ロジスティック回帰 ---
logreg = LogisticRegression(random_state=SEED).fit(X_train, y_train)
models["ロジスティック回帰"] = {"proba": logreg.predict_proba(X_test)[:, 1]}

In [ ]:
# --- 3. 決定木(深さを検証データで選ぶ) ---
best_depth, best_auc = None, -1.0
for depth in [2, 3, 4, 5, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X_train, y_train)
    auc = roc_auc_score(y_val, tree.predict_proba(X_val)[:, 1])
    print(f"max_depth={str(depth):>4} → 検証 AUC {auc:.3f}")
    if auc > best_auc:
        best_depth, best_auc = depth, auc

tree = DecisionTreeClassifier(max_depth=best_depth, random_state=SEED).fit(X_train, y_train)
models["決定木"] = {"proba": tree.predict_proba(X_test)[:, 1]}
print("採用した深さ:", best_depth)

# --- 4. ランダムフォレスト ---
forest = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(X_train, y_train)
models["ランダムフォレスト"] = {"proba": forest.predict_proba(X_test)[:, 1]}

### ニューラルネットワークの学習

隠れ層の構成を**検証データの AUC** で選び、採用した構成を `partial_fit`(1 エポックずつの学習)で
学習し直して、**学習損失と検証損失の推移**を記録します。
入力は 0/1 の 5 変数だけなので、標準化は不要です。

In [ ]:
candidates = [(8,), (16,), (32,), (16, 8)]
best_hidden, best_auc = None, -1.0
for hidden in candidates:
    clf = MLPClassifier(
        hidden_layer_sizes=hidden, learning_rate_init=0.01,
        max_iter=2000, random_state=SEED,
    ).fit(X_train, y_train)
    auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    print(f"隠れ層 {str(hidden):>8} → 検証 AUC {auc:.3f}")
    if auc > best_auc:
        best_hidden, best_auc = hidden, auc
print("採用した構成:", best_hidden)

In [ ]:
nn = MLPClassifier(
    hidden_layer_sizes=best_hidden, learning_rate_init=0.01,
    random_state=SEED,
)

n_epochs = 300
train_losses, val_losses = [], []
for epoch in range(n_epochs):
    nn.partial_fit(X_train, y_train, classes=[0, 1])
    train_losses.append(log_loss(y_train, nn.predict_proba(X_train)[:, 1], labels=[0, 1]))
    val_losses.append(log_loss(y_val, nn.predict_proba(X_val)[:, 1], labels=[0, 1]))

proba_val_nn = nn.predict_proba(X_val)[:, 1]
models["ニューラルネットワーク"] = {"proba": nn.predict_proba(X_test)[:, 1]}

plt.figure(figsize=(7, 3))
plt.plot(train_losses, label="学習損失")
plt.plot(val_losses, label="検証損失")
plt.title("ニューラルネットワークの損失の推移")
plt.xlabel("エポック")
plt.ylabel("Log Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. テストデータでの評価 — 5 モデルの比較

ここで初めてテストデータを使います。分類閾値は原則どおり **0.5** です
(多数派モデルのみ、定義上つねに多数派クラスを予測します)。

In [ ]:
def evaluate(y_true, proba, pred=None, threshold=0.5):
    if pred is None:
        pred = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1-score": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "Log Loss": log_loss(y_true, proba, labels=[0, 1]),
    }


results = pd.DataFrame({
    name: evaluate(y_test, m["proba"], m.get("pred"))
    for name, m in models.items()
}).T

results.round(3)

## 6. ニューラルネットワークの詳細評価

### 混同行列(閾値 0.5)

In [ ]:
proba_nn = models["ニューラルネットワーク"]["proba"]
pred_nn = (proba_nn >= 0.5).astype(int)

cm = confusion_matrix(y_test, pred_nn)
fig, ax = plt.subplots(figsize=(4, 3.4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                fontsize=16, color="black" if cm[i, j] < cm.max() * 0.6 else "white")
ax.set_xticks([0, 1], ["予測: 曇雨", "予測: 快晴"])
ax.set_yticks([0, 1], ["実測: 曇雨", "実測: 快晴"])
ax.set_title("混同行列(テストデータ)")
plt.tight_layout()
plt.show()

### ROC 曲線

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_nn)
auc_nn = roc_auc_score(y_test, proba_nn)

plt.figure(figsize=(4.5, 4))
plt.plot(fpr, tpr, label=f"NN (AUC = {auc_nn:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="当てずっぽう (AUC = 0.5)")
plt.xlabel("偽陽性率")
plt.ylabel("真陽性率")
plt.title("ROC 曲線(テストデータ)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 実測値と予測確率の比較

実測が快晴だった日と曇雨だった日で、モデルの出した予測確率の分布が分かれていれば、
確率として意味のある予測ができていることになります。

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(proba_nn[y_test == 0], bins=12, range=(0, 1), alpha=0.6,
         label="実測: 曇雨 (F=0)", color="steelblue")
plt.hist(proba_nn[y_test == 1], bins=12, range=(0, 1), alpha=0.6,
         label="実測: 快晴 (F=1)", color="darkorange")
plt.axvline(0.5, color="gray", linestyle="--")
plt.title("実測クラス別の予測確率の分布(テストデータ)")
plt.xlabel("予測確率")
plt.ylabel("日数")
plt.legend()
plt.tight_layout()
plt.show()

### F 地点の実測値と予測の時系列比較

In [ ]:
days = test["day"].to_numpy()

plt.figure(figsize=(10, 3.2))
plt.plot(days, proba_nn, color="seagreen", label="予測確率")
plt.scatter(days, y_test, color="black", s=18, zorder=3, label="実測 (0/1)")
plt.scatter(days[pred_nn != y_test], y_test[pred_nn != y_test],
            facecolors="none", edgecolors="crimson", s=90, zorder=4, label="外した日")
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.6)
plt.title("テスト期間の F 地点: 実測値と NN の予測確率")
plt.xlabel("day")
plt.ylabel("快晴の確率 / 実測")
plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

### 分類閾値の調整(検証データのみを使用)

**検証データ**で F1-score が最大になる閾値を探し、0.5 の場合とテスト性能を比べます。
テストデータは閾値選びには一切使いません。

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
f1_vals = [f1_score(y_val, (proba_val_nn >= t).astype(int), zero_division=0) for t in thresholds]
best_t = float(thresholds[int(np.argmax(f1_vals))])
print(f"検証データで F1 が最大になる閾値: {best_t:.2f} (F1 = {max(f1_vals):.3f})")

comparison = pd.DataFrame({
    "閾値 0.5": evaluate(y_test, proba_nn, threshold=0.5),
    f"閾値 {best_t:.2f}": evaluate(y_test, proba_nn, threshold=best_t),
}).T
comparison.round(3)

## 7. 32 種類の天気パターンの分析

A〜E は 0/1 なので、前日のパターンは全部で $2^5 = 32$ 通りです。
32 通りすべてを学習済み NN に入力して「翌日の F が快晴になる予測確率」を計算し、
**データ生成に使った理論上の確率**(生成規則をシグモイドに通した値)と並べます。

In [ ]:
bits = [(a, b, c, d, e)
        for a in (0, 1) for b in (0, 1) for c in (0, 1)
        for d in (0, 1) for e in (0, 1)]
patterns = pd.DataFrame(bits, columns=features)

patterns["predicted_probability"] = nn.predict_proba(patterns[features].to_numpy())[:, 1]
patterns["predicted_class"] = (patterns["predicted_probability"] >= 0.5).astype(int)
patterns["theoretical_probability"] = sigmoid(
    f_score_formula(patterns["A_lag1"], patterns["B_lag1"], patterns["C_lag1"],
                    patterns["D_lag1"], patterns["E_lag1"])
)

patterns = patterns.sort_values("predicted_probability", ascending=False).reset_index(drop=True)
patterns.to_csv("outputs/all_32_patterns.csv", index=False)
print("保存しました: outputs/all_32_patterns.csv")
print()
print("F が快晴になる予測確率の上位 10 パターン:")
print(patterns.head(10).to_string(index=False))

NN の予測確率と理論上の確率がどれだけ一致しているかを散布図で確認します。
点が対角線に乗っているほど、**NN がデータ生成の規則を復元できている**ことを意味します。

In [ ]:
r = np.corrcoef(patterns["theoretical_probability"], patterns["predicted_probability"])[0, 1]

plt.figure(figsize=(4.5, 4))
plt.scatter(patterns["theoretical_probability"], patterns["predicted_probability"],
            s=40, alpha=0.8)
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("理論上の確率(生成規則)")
plt.ylabel("NN の予測確率")
plt.title(f"32 パターンでの理論値との比較 (相関 r = {r:.3f})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"相関係数 r = {r:.3f}")
print(f"平均絶対誤差   = {np.abs(patterns['theoretical_probability'] - patterns['predicted_probability']).mean():.3f}")

## 8. 説明可能性

### Permutation Importance — どの地点を重視しているか

テストデータで各説明変数の値だけを**シャッフルして壊し**、ROC-AUC がどれだけ下がるかを測ります。
下がり方が大きい変数ほど、モデルの予測にとって重要です(30 回繰り返した平均)。

In [ ]:
def permutation_importance_auc(model, X, y, n_repeats=30, seed=SEED):
    base = roc_auc_score(y, model.predict_proba(X)[:, 1])
    rng = np.random.default_rng(seed)
    mean_drop, std_drop = [], []
    for j in range(X.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            rng.shuffle(Xp[:, j])
            drops.append(base - roc_auc_score(y, model.predict_proba(Xp)[:, 1]))
        mean_drop.append(np.mean(drops))
        std_drop.append(np.std(drops))
    return np.array(mean_drop), np.array(std_drop)


imp_mean, imp_std = permutation_importance_auc(nn, X_test, y_test)

plt.figure(figsize=(6.5, 3))
plt.bar(features, imp_mean, yerr=imp_std, capsize=4, color="steelblue")
plt.title("Permutation Importance(AUC の低下量、テストデータ)")
plt.ylabel("AUC の低下")
plt.axhline(0, color="gray", linewidth=0.8)
plt.tight_layout()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # ブラウザ版 matplotlib が保存時に出す非推奨警告を抑制
    plt.savefig("outputs/permutation_importance.png", dpi=120)
plt.show()
print("保存しました: outputs/permutation_importance.png")

for f_, m_ in sorted(zip(features, imp_mean), key=lambda x: -x[1]):
    print(f"{f_}: AUC 低下 {m_:+.3f}")

### 交互作用の確認 — A×C と B×E

データ生成規則には「A と C が両方快晴」「B と E が両方快晴」のときにスコアが上乗せされる
**交互作用**が入っています。32 パターン表を使い、2 地点の組み合わせごとに
NN の予測確率の平均(残り 3 地点の 8 通りをならした値)を見ます。

交互作用があるなら、「両方快晴」のマスの値が、単純な足し算から期待される値より高くなるはずです。

In [ ]:
def interaction_heatmap(ax, v1, v2, column, title):
    pivot = patterns.pivot_table(index=v1, columns=v2, values=column, aggfunc="mean")
    ax.imshow(pivot.to_numpy(), cmap="YlOrRd", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{pivot.iloc[i, j]:.2f}", ha="center", va="center", fontsize=13)
    site1, site2 = v1[0], v2[0]
    ax.set_xticks([0, 1], [f"{site2}=0", f"{site2}=1"])
    ax.set_yticks([0, 1], [f"{site1}=0", f"{site1}=1"])
    ax.set_title(title)


fig, axes = plt.subplots(2, 2, figsize=(8, 6.5))
interaction_heatmap(axes[0, 0], "A_lag1", "C_lag1", "predicted_probability", "A×C — NN の予測確率")
interaction_heatmap(axes[0, 1], "A_lag1", "C_lag1", "theoretical_probability", "A×C — 理論上の確率")
interaction_heatmap(axes[1, 0], "B_lag1", "E_lag1", "predicted_probability", "B×E — NN の予測確率")
interaction_heatmap(axes[1, 1], "B_lag1", "E_lag1", "theoretical_probability", "B×E — 理論上の確率")
plt.suptitle("交互作用の確認(各マスは残り 3 地点をならした平均確率)")
plt.tight_layout()
plt.show()

**やさしい言葉でまとめると**、上の分析はこう読めます。

- 棒グラフ(Permutation Importance)は「その地点の情報を消したら、予測がどれだけ下手になるか」を測ったものです。
  棒が高い地点ほど、モデルはその地点の前日の天気を頼りにしています。
- ヒートマップは「2 つの地点の組み合わせ」ごとの快晴予測の平均です。
  左上(両方曇雨)から右下(両方快晴)への値の増え方が**足し算以上**になっていれば、
  モデルは「2 地点そろって快晴のときは特に F が晴れやすい」という**組み合わせの効果**まで学習できたことになります。
- 右列の理論値(データを作ったときの本当の規則)と左列の NN の値が似ていれば、
  モデルは正解の仕組みをおおむね言い当てています。

## 9. 仮説の検証

### ブートストラップ法による ROC-AUC の 95% 信頼区間

テストデータから同じサイズの標本を復元抽出で 2000 回作り直し、AUC のばらつきを測ります。
**信頼区間の下限が 0.5 を上回っていれば**、「当てずっぽうより良い」ことが偶然とは考えにくい、と言えます。

In [ ]:
rng_boot = np.random.default_rng(SEED)
boot_aucs = []
for _ in range(2000):
    idx = rng_boot.integers(0, len(y_test), len(y_test))
    if y_test[idx].min() == y_test[idx].max():
        continue  # 片方のクラスしか含まれない標本では AUC を計算できない
    boot_aucs.append(roc_auc_score(y_test[idx], proba_nn[idx]))

ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])
print(f"NN の ROC-AUC = {auc_nn:.3f}")
print(f"ブートストラップ 95% 信頼区間: [{ci_low:.3f}, {ci_high:.3f}]  ({len(boot_aucs)} 標本)")

### 検証チェックリスト

最終レポートの 6 つの観点を、ここまでの結果からまとめて自動判定します。

In [ ]:
auc_lr = results.loc["ロジスティック回帰", "ROC-AUC"]
auc_major = results.loc["多数派モデル", "ROC-AUC"]
spread = patterns["predicted_probability"].max() - patterns["predicted_probability"].min()
top_imp = float(np.max(imp_mean))

checks = [
    ("1. NN の ROC-AUC が 0.5 を明確に上回る",
     ci_low > 0.5,
     f"AUC {auc_nn:.3f}、95%CI 下限 {ci_low:.3f} > 0.5"),
    ("2. 多数派モデルより指標が改善",
     (results.loc["ニューラルネットワーク", ["Accuracy", "F1-score", "ROC-AUC"]]
      > results.loc["多数派モデル", ["Accuracy", "F1-score", "ROC-AUC"]]).all()
     and results.loc["ニューラルネットワーク", "Log Loss"] < results.loc["多数派モデル", "Log Loss"],
     f"Accuracy {results.loc['ニューラルネットワーク', 'Accuracy']:.3f} vs "
     f"{results.loc['多数派モデル', 'Accuracy']:.3f} など(比較表参照)"),
    ("3. ロジスティック回帰より非線形モデルが改善",
     auc_nn > auc_lr,
     f"AUC: NN {auc_nn:.3f} vs ロジスティック回帰 {auc_lr:.3f} (差 {auc_nn - auc_lr:+.3f})"),
    ("4. 入力パターンによって予測確率が変化する",
     spread > 0.5,
     f"32 パターンの予測確率の幅 = {patterns['predicted_probability'].min():.2f}"
     f"〜{patterns['predicted_probability'].max():.2f}"),
    ("5. Permutation Importance で A〜E が寄与",
     top_imp > 0.02,
     f"最大の AUC 低下 {top_imp:.3f}(棒グラフ参照)"),
    ("6. 既知の生成規則をおおむね復元",
     r > 0.9,
     f"理論確率との相関 r = {r:.3f}"),
]

for label, ok, detail in checks:
    print(f"{'○' if ok else '×'} {label}")
    print(f"     {detail}")

## 10. 結論

このプロジェクトでは、「ある日の F 地点の天気は前日の A〜E 地点の天気パターンで決まる」という仮説を、
再現可能なダミーデータと 5 種類のモデルで検証しました。上のチェックリストが検証結果の要約です。

**仮説の中核部分(観点 1・2・4・5)は明確に支持されました。** 前日の A〜E には F を予測する情報が
確かに含まれており、NN は当てずっぽう(AUC 0.5)や多数派モデルを大きく上回りました。

一方、この設定(シード 42・約 1 年分)では **観点 3 と 6 は×になります**。これは手法の誤りではなく、
次のように解釈できます。

- **観点 3(線形モデルに勝てない)**: 生成規則は交互作用を含むものの、`1.2 × (A+…+E)` という
  線形の主効果が支配的です。AUC は「並べ方」の指標なので、主効果だけでもかなり正しく並べられる
  ロジスティック回帰が強く、非線形モデルの出番は小さくなります。非線形の優位性は
  「データにどれだけ強い交互作用があるか」に依存する、というのがここでの学びです。
- **観点 6(規則の復元が不完全)**: 学習に使えるのは約 250 日分で、32 パターンのうち出現回数が
  数回しかないものもあります。データ量を数年分に増やすと理論確率との相関は大きく改善します
  (発展課題を参照)。

### ⚠️ 重要な注意

**この分析は「分析方法が正しく動くことの確認」です。**
データは既知の規則から人工的に生成したものであり、ここでの結果は
**現実の天気に関する仮説が実証されたことを意味しません**。
現実のデータに適用する際は、さらに次の点に注意が必要です。

- テストデータが約 54 日分と少なく、指標のばらつきが大きい(信頼区間の広さに表れています)
- 乱数シード 1 通り・1 年分だけの結果である(複数シード・複数年での再現確認が望ましい)
- 現実の天気は 6 地点の二値変数よりはるかに複雑で、観測されない要因(気圧配置など)の影響を受ける

### 発展課題

- データを 3〜10 年分に増やし、性能と信頼区間がどう変わるか調べる
- 2 日前・3 日前のラグ変数を加えて予測が改善するか試す
- ロジスティック回帰に交互作用項(A×C、B×E)を明示的に加えると NN に追いつくか確認する